<a id="top"></a>
# BB84 gate round-trip baseline

Compare selected X–X and H–H round trips with same-state no-gate controls
across the four BB84 states. **Run All** produces one comparison plot.

> **Hardware is explicit opt-in.** The committed `RUN_MODE = "ideal"` stays
> on local Aer and never reads credentials or submits a job. Change it to
> `"hardware"` only after reviewing the hardware settings and local
> `apikey.json`.


## Configuration

Edit only this first code cell for normal runs. Remove `"X"` or `"H"`
from `TEST_GATES` to run a single probe.


In [ ]:
RUN_MODE = "ideal"  # "ideal" for local Aer or "hardware" for IBM Runtime
TEST_GATES = ("X", "H")

# Local Aer noise. These settings are ignored in hardware mode.
ENABLE_SIM_NOISE = True
SIM_GATE_ERROR_RATE = 0.01
SIM_READOUT_ERROR_RATE = 0.02

SHOTS = 4096
SEED = 20261028

# Hardware only. Running in hardware mode authenticates and submits a job.
API_KEY_FILE = "apikey.json"
QPU = "ibm_fez"
PHYSICAL_QUBIT = 0
HARDWARE_OPTIMIZATION_LEVEL = 1


## Run All: expected plot

Each bar is the mean wrong-outcome percentage across `|0⟩`, `|1⟩`, `|+⟩`,
and `|−⟩`. The control and round trip use the same preparations and
measurement bases.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error

if RUN_MODE not in {"ideal", "hardware"}:
    raise ValueError("RUN_MODE must be 'ideal' or 'hardware'")
if not TEST_GATES or any(gate not in {"X", "H"} for gate in TEST_GATES):
    raise ValueError("TEST_GATES must contain X, H, or both")
if type(SHOTS) is not int or SHOTS < 1:
    raise ValueError("SHOTS must be a positive integer")
if any(not 0 <= rate <= 1 for rate in (SIM_GATE_ERROR_RATE, SIM_READOUT_ERROR_RATE)):
    raise ValueError("Simulation error rates must be between 0 and 1")

if RUN_MODE == "ideal":
    noise_model = None
    if ENABLE_SIM_NOISE:
        noise_model = NoiseModel()
        if SIM_GATE_ERROR_RATE:
            noise_model.add_all_qubit_quantum_error(
                depolarizing_error(SIM_GATE_ERROR_RATE, 1), ["x", "h"]
            )
        if SIM_READOUT_ERROR_RATE:
            flip = SIM_READOUT_ERROR_RATE
            noise_model.add_all_qubit_readout_error(
                ReadoutError([[1 - flip, flip], [flip, 1 - flip]])
            )
    backend = AerSimulator(noise_model=noise_model)
else:
    from qiskit.transpiler import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService
    from qiskit_ibm_runtime.executor_sampler import Sampler

    credentials = json.loads(Path(API_KEY_FILE).read_text(encoding="utf-8"))
    token = credentials.get("apikey")
    if not isinstance(token, str) or not token.strip():
        raise ValueError(f"{API_KEY_FILE} must contain a non-empty 'apikey' field")
    service_options = {"channel": "ibm_quantum_platform", "token": token.strip()}
    if credentials.get("crn"):
        service_options["instance"] = credentials["crn"]
    backend = QiskitRuntimeService(**service_options).backend(QPU)

gate_methods = {"X": "x", "H": "h"}
states = (("Z", 0), ("Z", 1), ("X", 0), ("X", 1))
circuits, cases = [], []
for gate_name in TEST_GATES:
    for basis, bit in states:
        for run_type in ("Control", "Gate round trip"):
            circuit = QuantumCircuit(1, 1)
            if bit:
                circuit.x(0)
            if basis == "X":
                circuit.h(0)
            if run_type == "Gate round trip":
                circuit.barrier()
                getattr(circuit, gate_methods[gate_name])(0)
                circuit.barrier()
                getattr(circuit, gate_methods[gate_name])(0)
                circuit.barrier()
            if basis == "X":
                circuit.h(0)
            circuit.measure(0, 0)
            circuits.append(circuit)
            cases.append((gate_name, run_type, bit))

if RUN_MODE == "ideal":
    counts_by_circuit = backend.run(
        circuits, shots=SHOTS, seed_simulator=SEED
    ).result().get_counts()
    if not isinstance(counts_by_circuit, list):
        counts_by_circuit = [counts_by_circuit]
else:
    pass_manager = generate_preset_pass_manager(
        backend=backend,
        optimization_level=HARDWARE_OPTIMIZATION_LEVEL,
        initial_layout=[PHYSICAL_QUBIT],
    )
    job = Sampler(mode=backend).run(pass_manager.run(circuits), shots=SHOTS)
    print("IBM Runtime job ID:", job.job_id())
    counts_by_circuit = []
    for result in job.result():
        registers = tuple(result.data.keys())
        if len(registers) != 1:
            raise RuntimeError(f"Expected one measurement register, found {registers}")
        counts_by_circuit.append(getattr(result.data, registers[0]).get_counts())

if len(counts_by_circuit) != len(cases):
    raise RuntimeError("Backend result count does not match the circuit count")
rates = {(gate, run): [] for gate in TEST_GATES for run in ("Control", "Gate round trip")}
for (gate, run, expected), counts in zip(cases, counts_by_circuit):
    shots_observed = sum(counts.values())
    wrong = sum(
        count
        for outcome, count in counts.items()
        if int(outcome.replace(" ", ""), 2) != expected
    )
    rates[(gate, run)].append(100 * wrong / shots_observed)
mean_rates = {key: sum(values) / len(values) for key, values in rates.items()}

positions = list(range(len(TEST_GATES)))
width = 0.36
fig, ax = plt.subplots(figsize=(7, 4))
control_bars = ax.bar(
    [x - width / 2 for x in positions],
    [mean_rates[(gate, "Control")] for gate in TEST_GATES],
    width,
    label="Control",
)
round_trip_bars = ax.bar(
    [x + width / 2 for x in positions],
    [mean_rates[(gate, "Gate round trip")] for gate in TEST_GATES],
    width,
    label="Gate round trip",
)
mode_label = (
    "noisy local Aer"
    if RUN_MODE == "ideal" and ENABLE_SIM_NOISE
    else "noiseless local Aer"
    if RUN_MODE == "ideal"
    else f"{QPU} hardware"
)
ax.set_title(f"BB84 {'/'.join(TEST_GATES)} round-trip baseline ({mode_label})")
ax.set_ylabel("Mean wrong outcomes (%)")
ax.set_xlabel("Tested logical gate")
ax.set_xticks(positions, TEST_GATES)
ax.legend()
ax.grid(axis="y", alpha=0.25)
for bars in (control_bars, round_trip_bars):
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
fig.tight_layout()
plt.show()


## Interpretation and sources

Within each gate, compare **Gate round trip** with **Control**. Any excess is
an end-to-end circuit effect, not isolated single-gate infidelity or a QKD
security result. The synthetic Aer noise also affects X/H instructions used
for state preparation and measurement-basis changes. Hardware results also
include transpilation, routing, preparation, and readout effects.

Sources: [BB84 state encoding](https://arxiv.org/abs/2003.06557), Qiskit Aer
[`NoiseModel`](https://qiskit.github.io/qiskit-aer/apidocs/aer_noise.html),
and IBM's [Sampler and transpilation guide](https://quantum.cloud.ibm.com/docs/en/guides/sampler-options).
